In [52]:
sample_strings = [
    "00000000001100010000000010110011",
    "01000000100000101000001000110011",
    "00000000100101000000001110110011",
    "01000000010100110000010010110011",
    "00000000010000010010001010000011",
    "11111111100000101010001100000011",
    "00000000110001001010010000000011",
    "00000000010100010010010000100011",
    "11111110011000101010100000100011",
    "00000000101001001010011000100011"
]

In [53]:
register_file = ["0"*32] * 32

In [54]:
memory = ["0"*8] * 4096 # 4096 bytes of memory, each byte is 8 bits

## Buffer Registers:

In [55]:
RF_EX_buffer = {
    "A": None,
    "B": None,
    "rs1": None,
    "rs2": None,
    "rd": None,
    "funct7": None,
    "isAdd": None,
    "isSub": None,
    "isLd": None,
    "isSt": None,
    "doWB": None
}

EX_MEM_buffer = {
    "alu_result": None,
    "rs2_val": None,
    "rd": None,
    "isLd": None,
    "isSt": None,
    "doWB": None
}

MEM_WB_buffer = {
    "writeback_value": None,
    "rs2_val": None,
    "rd": None,
    "isLd": None,
    "isSt": None,
    "doWB": None
}

## Register Fetch / Instruction decode stage

In [56]:
# Core Register Fetch / Instruction decode

def get_opcode(instruction):
    return instruction[25:]

def get_funct3(instruction):
    return instruction[17:20]

def get_funct7(instruction):
    return instruction[0:7]

def get_rs1(instruction):
    register_index_string = instruction[12:17]
    register_value = register_file[int(register_index_string, 2)]
    return register_value

def get_rs2(instruction):
    register_index_string = instruction[7:12]
    register_value = register_file[int(register_index_string, 2)]
    return register_value

def get_rd(instruction):
    return instruction[20:25]

In [57]:
# Control Unit

def control_unit(instruction):
    opcode = get_opcode(instruction)
    funct3 = get_funct3(instruction)
    funct7 = get_funct7(instruction)

    isAdd = (opcode == "0110011" and funct3 == "000" and funct7 == "0000000")
    isSub = (opcode == "0110011" and funct3 == "000" and funct7 == "0100000")
    
    isLd = (opcode == "0000011")
    isSt = (opcode == "0100011")

    doWB = isAdd or isSub or isLd

    return {
        "isAdd": isAdd,
        "isSub": isSub,
        "isLd": isLd,
        "isSt": isSt,
        "doWB": doWB
    }

In [58]:
# main function of instruction decode stage

def instruction_decode(instruction):
    RF_EX_buffer["A"] = get_rs1(instruction)
    RF_EX_buffer["B"] = get_rs2(instruction)
    RF_EX_buffer["rd"] = get_rd(instruction)

    control_output = control_unit(instruction)
    RF_EX_buffer["isAdd"] = control_output["isAdd"]
    RF_EX_buffer["isSub"] = control_output["isSub"]
    RF_EX_buffer["isLd"] = control_output["isLd"]
    RF_EX_buffer["isSt"] = control_output["isSt"]
    RF_EX_buffer["doWB"] = control_output["doWB"]

In [59]:
# Branch unit implementation

## Execute Stage

In [60]:
# A, B resolution for ALU

def input_resolve():
    A = RF_EX_buffer["A"]

    if RF_EX_buffer["isLd"]:
        B = int(RF_EX_buffer["funct7"] + RF_EX_buffer["rs2"], 2)
    elif RF_EX_buffer["isSt"]:
        B = int(RF_EX_buffer["funct7"] + RF_EX_buffer["rd"], 2)
    else:
        B = RF_EX_buffer["B"]

    return A, B

In [61]:
# Execute stage

def Adder(A, B, isAdd, isSub, isLd, isSt):
    if isAdd or isLd or isSt:
        return A + B
    if isSub:
        return A - B

def Multiplier(A, B, isMul):
    return 0

def Divider(A, B, isDiv, isMod):
    return 0

def ShiftUnit(A, B, isLsl, isLsr, isAsr):
    return 0

def LogicalUnit(A, B, isAnd, isOr, isNot):
    return 0

def isMov():
    return 0


In [62]:
def execute():
    EX_MEM_buffer["rd"] = RF_EX_buffer["rd"]

    EX_MEM_buffer["isLd"] = RF_EX_buffer["isLd"]
    EX_MEM_buffer["isSt"] = RF_EX_buffer["isSt"]

    EX_MEM_buffer["A"] = RF_EX_buffer["A"]
    EX_MEM_buffer["B"] = RF_EX_buffer["B"]

    EX_MEM_buffer["alu_result"] = Adder(RF_EX_buffer["A"], RF_EX_buffer["B"], RF_EX_buffer["isAdd"], RF_EX_buffer["isSub"], RF_EX_buffer["isLd"], RF_EX_buffer["isSt"])

    EX_MEM_buffer["doWB"] = RF_EX_buffer["doWB"]

## Memory Stage

In [63]:
def memory_access():
    if EX_MEM_buffer["isLd"]:
        MEM_WB_buffer["writeback_value"] = memory[EX_MEM_buffer["alu_result"]]

    elif EX_MEM_buffer["isSt"]:
        memory[EX_MEM_buffer["alu_result"]] = MEM_WB_buffer["rs2_val"]

    MEM_WB_buffer["doWB"] = EX_MEM_buffer["doWB"]

## Register Write Back Stage

In [64]:
def write_back():
    if MEM_WB_buffer["doWB"]:
        # convert writeback value to binary string and store in register file
        register_file[int(MEM_WB_buffer["rd"], 2)] = format(MEM_WB_buffer["writeback_value"], '032b')

## Testing

In [65]:
register_file[3] = "00000000000000000000000000000101" # x3 = 5
register_file[2] = "00000000000000000000000000001010" # x2 = 10

test_instruction = "00000000001100010000000010110011" # add x1, x2, x3

instruction_decode(test_instruction)
execute()
memory_access()

print("Register file after execution:")
for i in range(len(register_file)):
    print(f"x{i}: {register_file[i]}")

Register file after execution:
x0: 00000000000000000000000000000000
x1: 00000000000000000000000000000000
x2: 00000000000000000000000000001010
x3: 00000000000000000000000000000101
x4: 00000000000000000000000000000000
x5: 00000000000000000000000000000000
x6: 00000000000000000000000000000000
x7: 00000000000000000000000000000000
x8: 00000000000000000000000000000000
x9: 00000000000000000000000000000000
x10: 00000000000000000000000000000000
x11: 00000000000000000000000000000000
x12: 00000000000000000000000000000000
x13: 00000000000000000000000000000000
x14: 00000000000000000000000000000000
x15: 00000000000000000000000000000000
x16: 00000000000000000000000000000000
x17: 00000000000000000000000000000000
x18: 00000000000000000000000000000000
x19: 00000000000000000000000000000000
x20: 00000000000000000000000000000000
x21: 00000000000000000000000000000000
x22: 00000000000000000000000000000000
x23: 00000000000000000000000000000000
x24: 00000000000000000000000000000000
x25: 000000000000000000000000